In [ ]:
import pandas as pd
raw = pd.read_parquet("dados/clique_economia_mensal_2024_2026.parquet")
raw["id_produto"] = raw["id_produto"].astype(str)
print(raw["preco_regular"].head(3).tolist(), raw["preco_regular"].dtype)
raw["preco_regular"] = pd.to_numeric(raw["preco_regular"].astype(str).str.replace(",", ".", regex=False), errors="coerce")

c37 = pd.read_csv("dados/ncm_rastreabilidade_37.csv", dtype=str)
c13 = pd.read_csv("dados/catalogo_expansao_13.csv", dtype=str)
cat = pd.concat([c37[["grupo","id_produto","descricao"]], c13[["grupo","id_produto","descricao"]]], ignore_index=True)
print("catalogo:", len(cat), cat["id_produto"].nunique())

x = raw[raw["id_produto"].isin(cat["id_produto"])]
pos = x[x["preco_regular"] > 0]
print("linhas por EAN:", len(x), "| positivas:", len(pos))
k = ["id_produto","id_empresa","endereco_completo","data_pesquisa"]
print("repetidas (todas as linhas da chave):", pos.duplicated(k, keep=False).sum(), "| excedentes:", pos.duplicated(k).sum())
print("sem id_empresa:", pos["id_empresa"].isna().sum(), "| vazio:", (pos["id_empresa"].astype(str).str.strip()=="").sum())

In [ ]:
k = ["id_produto","id_empresa","endereco_completo","data_pesquisa"]
d = pos[pos.duplicated(k, keep=False)].copy()
g = d.groupby(k, dropna=False)
resumo = g.agg(n=("preco_regular","size"), n_precos=("preco_regular","nunique"), pmin=("preco_regular","min"), pmax=("preco_regular","max"), meses=("__mes","nunique"), arqs=("__arq","nunique")).reset_index()
print("chaves repetidas:", len(resumo))
print("distribuicao de linhas por chave:"); print(resumo["n"].value_counts().sort_index().to_string())
print("chaves com preco IGUAL:", (resumo["n_precos"]==1).sum(), "| com preco DIFERENTE:", (resumo["n_precos"]>1).sum())
conf = resumo[resumo["n_precos"]>1].copy()
conf["dif_pct"] = (conf["pmax"]-conf["pmin"])/conf["pmin"]*100
print("conflitos: diferenca percentual"); print(conf["dif_pct"].describe().round(2).to_string())
print("mesmo __arq nas duplicatas (arqs==1):", (resumo["arqs"]==1).sum(), "| em arquivos diferentes:", (resumo["arqs"]>1).sum())
print("por grupo:"); print(d.merge(cat[["id_produto","grupo"]], on="id_produto").groupby("grupo").size().sort_values(ascending=False).to_string())
print(conf.sort_values("dif_pct", ascending=False).head(8).to_string(index=False))

In [ ]:
k = ["id_produto","id_empresa","endereco_completo","data_pesquisa"]
d = pos[pos.duplicated(k, keep=False)].copy()
for c in ["preco_promocao","preco_fidelidade","preco_atacado","preco_atacado_qtd"]:
    d[c] = d[c].astype(str)
par = d.groupby(k, dropna=False).agg(
    mes_a=("__mes","min"), mes_b=("__mes","max"),
    prom=("preco_promocao","nunique"), fid=("preco_fidelidade","nunique"),
    atac=("preco_atacado","nunique"), atacq=("preco_atacado_qtd","nunique")).reset_index()
par["dif_meses"] = (pd.PeriodIndex(par["mes_b"], freq="M") - pd.PeriodIndex(par["mes_a"], freq="M")).map(lambda x: x.n)
print("dif de meses entre os dois arquivos:"); print(par["dif_meses"].value_counts().sort_index().to_string())
print("pares com promocao diferente:", (par["prom"]>1).sum(), "| fidelidade:", (par["fid"]>1).sum(), "| atacado:", (par["atac"]>1).sum(), "| atacado_qtd:", (par["atacq"]>1).sum())
print("exemplos (data_pesquisa, mes_a, mes_b):"); print(par[["data_pesquisa","mes_a","mes_b"]].head(8).to_string(index=False))
print("pares por mes_a (top 10):"); print(par["mes_a"].value_counts().head(10).to_string())
print("dtype data_pesquisa:", pos["data_pesquisa"].dtype)

In [ ]:
p = pos.copy()
p["data"] = pd.to_datetime(p["data_pesquisa"], errors="coerce")
print("datas invalidas:", p["data"].isna().sum(), "| min:", p["data"].min(), "| max:", p["data"].max())
p["mes_data"] = p["data"].dt.to_period("M").astype(str)
dif = p[p["mes_data"] != p["__mes"].astype(str)]
print("linhas com mes_data != __mes:", len(dif), "de", len(p))
print(dif.groupby(["__mes","mes_data"]).size().sort_values(ascending=False).head(15).to_string())
k = ["id_produto","id_empresa","endereco_completo","data_pesquisa"]
u = p.sort_values("__mes").drop_duplicates(k, keep="first")
print("apos deduplicar:", len(u))
print("dif mes_data != __mes apos deduplicar:", (u["mes_data"] != u["__mes"].astype(str)).sum())
print("dia da data_pesquisa (top 8):"); print(p["data"].dt.day.value_counts().head(8).to_string())

In [ ]:
u["data"] = pd.to_datetime(u["data_pesquisa"])
f = u.groupby("__mes").agg(data_min=("data","min"), data_max=("data","max"), dias=("data","nunique"), obs=("data","size")).reset_index()
f["span_dias"] = (f["data_max"] - f["data_min"]).dt.days
print(f.to_string(index=False))
print("span_dias: mediana", f["span_dias"].median(), "| max", f["span_dias"].max())
v = u.groupby(["id_produto","endereco_completo","__mes"])["data"].nunique()
print("SKU x loja x arquivo com mais de 1 data:", (v>1).sum(), "de", len(v))
print(v.value_counts().sort_index().to_string())

In [ ]:
r = raw.copy()
r["data"] = pd.to_datetime(r["data_pesquisa"], errors="coerce")
g = r.groupby("__mes").agg(data_min=("data","min"), data_max=("data","max"), linhas=("data","size"), arquivos=("__arq","nunique")).reset_index()
print(g.to_string(index=False))
print("meses distintos na base:", r["__mes"].nunique())
print(r[r["__mes"]=="2024-05"]["__arq"].unique())

In [ ]:
u["data"] = pd.to_datetime(u["data_pesquisa"])
m_antes = pos.groupby("id_produto")["__mes"].nunique()
m_depois = u.groupby("id_produto")["__mes"].nunique()
datas = u.groupby("id_produto")["data"].nunique()
t = cat.set_index("id_produto")[["grupo"]].join(m_antes.rename("meses_antes")).join(m_depois.rename("meses_depois")).join(datas.rename("rodadas_datas"))
t["perdeu"] = t["meses_antes"] - t["meses_depois"]
print("SKUs que perdem meses apos deduplicar:", (t["perdeu"]>0).sum(), "de", len(t))
print(t["perdeu"].value_counts().sort_index().to_string())
print(t.groupby("grupo")[["meses_antes","meses_depois","rodadas_datas"]].median().to_string())
print(t.sort_values("perdeu", ascending=False).head(8).to_string())

In [ ]:
perdidos = t[t["perdeu"] > 0].index
saiu = []
for e in perdidos:
    antes = set(pos.loc[pos["id_produto"] == e, "__mes"].astype(str))
    depois = set(u.loc[u["id_produto"] == e, "__mes"].astype(str))
    saiu.append(tuple(sorted(antes - depois)))
print(pd.Series(saiu).value_counts().to_string())
print("linhas de 2024-05 que sobram em u:", (u["__mes"].astype(str) == "2024-05").sum())
print("linhas de 2024-03 em u:", (u["__mes"].astype(str) == "2024-03").sum(), "| em pos:", (pos["__mes"].astype(str) == "2024-03").sum())

In [ ]:
"""Limpeza da base de precos do Clique Economia para os SKUs do catalogo.

Regras (verificadas em 07/11/2026 sobre os 50 SKUs do catalogo):
1. Filtro: preco_regular > 0 (preco zerado = condicao inexistente, nao e preco).
2. Deduplicacao pela chave id_produto + id_empresa + endereco_completo + data_pesquisa,
   mantendo a linha do primeiro arquivo (__mes mais antigo). Nos 281 pares encontrados,
   todos os precos (regular, promocao, fidelidade, atacado, atacado_qtd) eram identicos
   e os arquivos eram consecutivos.
3. __mes e o eixo da serie (cada arquivo e uma rodada de coleta de 0 a 1 dia);
   data_pesquisa e a data real da observacao.
4. Meses de cobertura = __mes distintos apos deduplicar. Nao usar contagem de datas.
"""
from pathlib import Path

import pandas as pd

CHAVE_DUP = ["id_produto", "id_empresa", "endereco_completo", "data_pesquisa"]


def carregar_base(caminho: str = "dados/clique_economia_mensal_2024_2026.parquet") -> pd.DataFrame:
    """Le o parquet, mantem codigos como texto e converte preco_regular para numero."""
    raw = pd.read_parquet(Path(caminho))
    raw["id_produto"] = raw["id_produto"].astype(str)
    raw["preco_regular"] = pd.to_numeric(
        raw["preco_regular"].astype(str).str.replace(",", ".", regex=False), errors="coerce"
    )
    return raw


def carregar_catalogo(
    caminho_37: str = "dados/ncm_rastreabilidade_37.csv",
    caminho_13: str = "dados/catalogo_expansao_13.csv",
) -> pd.DataFrame:
    """Une os 37 SKUs originais e os 13 da expansao (colunas grupo, id_produto, descricao)."""
    c37 = pd.read_csv(caminho_37, dtype=str)
    c13 = pd.read_csv(caminho_13, dtype=str)
    cols = ["grupo", "id_produto", "descricao"]
    return pd.concat([c37[cols], c13[cols]], ignore_index=True)


def limpar(raw: pd.DataFrame, catalogo: pd.DataFrame) -> pd.DataFrame:
    """Vinculo exato por EAN, preco regular positivo e deduplicacao. Retorna observacoes unicas."""
    x = raw[raw["id_produto"].isin(catalogo["id_produto"])]
    pos = x[x["preco_regular"] > 0].copy()
    pos["data"] = pd.to_datetime(pos["data_pesquisa"], errors="coerce")
    unicas = pos.sort_values("__mes").drop_duplicates(CHAVE_DUP, keep="first")
    return unicas.merge(catalogo[["id_produto", "grupo"]], on="id_produto", how="left")


if __name__ == "__main__":
    base = carregar_base()
    cat = carregar_catalogo()
    obs = limpar(base, cat)
    print("observacoes unicas:", len(obs))

In [ ]:
import pandas as pd
from mercado.base_precos import carregar_base, carregar_catalogo, limpar

raw = carregar_base()
cat = carregar_catalogo()
obs = limpar(raw, cat)
print("observacoes unicas:", len(obs))

antes = raw[raw["id_produto"].isin(cat["id_produto"]) & (raw["preco_regular"] > 0)]
print("antes (positivas pelo EAN):", len(antes), "| depois (unicas):", len(obs))

obs["__mes"] = obs["__mes"].astype(str)
cob = obs.groupby("id_produto").agg(
    registros=("preco_regular", "size"),
    meses=("__mes", "nunique"),
    redes=("rede", "nunique"),
    lojas=("endereco_completo", "nunique"),
    primeira=("__mes", "min"),
    ultima=("__mes", "max"),
).reset_index()
cob = cat.merge(cob, on="id_produto", how="left")
print(cob.groupby("grupo")[["registros","meses","redes","lojas"]].agg(["sum","median"]).round(1).to_string())
print(cob.sort_values("registros").head(8)[["grupo","id_produto","descricao","registros","meses","redes","lojas","ultima"]].to_string(index=False))

In [ ]:
import importlib, sys
import mercado.base_precos as m
importlib.reload(m)
print(m.__file__)
print([n for n in dir(m) if not n.startswith("_")])
print("modulos mercado em cache:", [k for k in sys.modules if k.startswith("mercado")])

In [ ]:
"""Limpeza da base de precos do Clique Economia para os SKUs do catalogo.

Regras (verificadas em 07/11/2026 sobre os 50 SKUs do catalogo):
1. Filtro: preco_regular > 0 (preco zerado = condicao inexistente, nao e preco).
2. Deduplicacao pela chave id_produto + id_empresa + endereco_completo + data_pesquisa,
   mantendo a linha do primeiro arquivo (__mes mais antigo). Nos 281 pares encontrados,
   todos os precos (regular, promocao, fidelidade, atacado, atacado_qtd) eram identicos
   e os arquivos eram consecutivos.
3. __mes e o eixo da serie (cada arquivo e uma rodada de coleta de 0 a 1 dia);
   data_pesquisa e a data real da observacao.
4. Meses de cobertura = __mes distintos apos deduplicar. Nao usar contagem de datas.
"""
from pathlib import Path

import pandas as pd

CHAVE_DUP = ["id_produto", "id_empresa", "endereco_completo", "data_pesquisa"]


def carregar_base(caminho: str = "dados/clique_economia_mensal_2024_2026.parquet") -> pd.DataFrame:
    """Le o parquet, mantem codigos como texto e converte preco_regular para numero."""
    raw = pd.read_parquet(Path(caminho))
    raw["id_produto"] = raw["id_produto"].astype(str)
    raw["preco_regular"] = pd.to_numeric(
        raw["preco_regular"].astype(str).str.replace(",", ".", regex=False), errors="coerce"
    )
    return raw


def carregar_catalogo(
    caminho_37: str = "dados/ncm_rastreabilidade_37.csv",
    caminho_13: str = "dados/catalogo_expansao_13.csv",
) -> pd.DataFrame:
    """Une os 37 SKUs originais e os 13 da expansao (colunas grupo, id_produto, descricao)."""
    c37 = pd.read_csv(caminho_37, dtype=str)
    c13 = pd.read_csv(caminho_13, dtype=str)
    cols = ["grupo", "id_produto", "descricao"]
    return pd.concat([c37[cols], c13[cols]], ignore_index=True)


def limpar(raw: pd.DataFrame, catalogo: pd.DataFrame) -> pd.DataFrame:
    """Vinculo exato por EAN, preco regular positivo e deduplicacao. Retorna observacoes unicas."""
    x = raw[raw["id_produto"].isin(catalogo["id_produto"])]
    pos = x[x["preco_regular"] > 0].copy()
    pos["data"] = pd.to_datetime(pos["data_pesquisa"], errors="coerce")
    unicas = pos.sort_values("__mes").drop_duplicates(CHAVE_DUP, keep="first")
    return unicas.merge(catalogo[["id_produto", "grupo"]], on="id_produto", how="left")


if __name__ == "__main__":
    base = carregar_base()
    cat = carregar_catalogo()
    obs = limpar(base, cat)
    print("observacoes unicas:", len(obs))

In [ ]:
import pandas as pd
from mercado.base_precos import carregar_base, carregar_catalogo, limpar

raw = carregar_base()
cat = carregar_catalogo()
obs = limpar(raw, cat)
print("observacoes unicas:", len(obs))

antes = raw[raw["id_produto"].isin(cat["id_produto"]) & (raw["preco_regular"] > 0)]
print("antes (positivas pelo EAN):", len(antes), "| depois (unicas):", len(obs))

obs["__mes"] = obs["__mes"].astype(str)
cob = obs.groupby("id_produto").agg(
    registros=("preco_regular", "size"),
    meses=("__mes", "nunique"),
    redes=("rede", "nunique"),
    lojas=("endereco_completo", "nunique"),
    primeira=("__mes", "min"),
    ultima=("__mes", "max"),
).reset_index()
cob = cat.merge(cob, on="id_produto", how="left")
print(cob.groupby("grupo")[["registros","meses","redes","lojas"]].agg(["sum","median"]).round(1).to_string())
print(cob.sort_values("registros").head(8)[["grupo","id_produto","descricao","registros","meses","redes","lojas","ultima"]].to_string(index=False))

In [ ]:
cob_out = cob.copy()
cob_out["registros_antes"] = cob_out["id_produto"].map(antes.groupby("id_produto").size()).fillna(0).astype(int)
cob_out["duplicatas_removidas"] = cob_out["registros_antes"] - cob_out["registros"]
cob_out.to_csv("raw_data/cobertura_por_sku.csv", index=False)
print(cob_out.shape)
print("registros_antes soma:", cob_out["registros_antes"].sum(), "| registros soma:", cob_out["registros"].sum(), "| removidas:", cob_out["duplicatas_removidas"].sum())
print(cob_out[cob_out["duplicatas_removidas"] > 0][["grupo","descricao","registros_antes","registros","duplicatas_removidas"]].head(5).to_string(index=False))
print(open("raw_data/cobertura_por_sku.csv").readline())

In [ ]:
g = obs.groupby("grupo").agg(
    registros=("preco_regular", "size"),
    skus=("id_produto", "nunique"),
    meses=("__mes", "nunique"),
    redes=("rede", "nunique"),
    lojas=("endereco_completo", "nunique"),
    primeira=("__mes", "min"),
    ultima=("__mes", "max"),
).reset_index()
print(g.to_string(index=False))
print("total distinto -> redes:", obs["rede"].nunique(), "| lojas:", obs["endereco_completo"].nunique(), "| meses:", obs["__mes"].nunique())

In [ ]:
ph = ["1000000","1000001","1000002","999999","999998"]
x = raw[raw["id_produto"].isin(ph)].copy()
x["data"] = pd.to_datetime(x["data_pesquisa"], errors="coerce")
print(x.groupby(["id_produto","descricao"]).agg(
    linhas=("preco_regular","size"),
    positivas=("preco_regular", lambda s: (s>0).sum()),
    meses=("__mes","nunique"),
    primeiro=("__mes","min"),
    ultimo=("__mes","max"),
    lojas=("endereco_completo","nunique")).to_string())

In [ ]:
ref = cat[cat["grupo"]=="Refrigerante 2 L"][["id_produto","descricao"]]
print(ref.to_string(index=False))
ph = {"1000000":"COCA","1000001":"GUARANA ANTARTICA","1000002":"PEPSI","999999":"CINI"}
r = raw[raw["preco_regular"]>0].copy()
for pid, marca in ph.items():
    a = r[r["id_produto"]==pid]
    e = r[r["id_produto"].isin(ref["id_produto"]) & r["descricao"].str.upper().str.contains(marca.split()[0], na=False)]
    print(marca, "| placeholder:", a["__mes"].min(), "a", a["__mes"].max(), "n=", len(a), "mediana=", a["preco_regular"].median(),
          "| EAN(s):", sorted(e["id_produto"].unique()), e["__mes"].min(), "a", e["__mes"].max(), "n=", len(e), "mediana=", e["preco_regular"].median())

In [ ]:
pares = {"COCA":("1000000","7894900027013"), "GUARANA ANTARTICA":("1000001","7891991001342"),
         "PEPSI":("1000002","7892840800246"), "CINI":("999999","7896188403077")}
r = raw[raw["preco_regular"]>0]
for marca,(ph,ean) in pares.items():
    a = r[r["id_produto"]==ph]; e = r[r["id_produto"]==ean]
    comuns = sorted(set(a["__mes"].astype(str)) & set(e["__mes"].astype(str)))
    print(marca, "| placeholder:", a["__mes"].min(), "a", a["__mes"].max(), "n=", len(a), "med=", a["preco_regular"].median(),
          "| EAN:", e["__mes"].min(), "a", e["__mes"].max(), "n=", len(e), "med=", e["preco_regular"].median(),
          "| meses em comum:", comuns)

In [ ]:
pares = {"COCA":("1000000","7894900027013"), "GUARANA ANTARTICA":("1000001","7891991001342"),
         "PEPSI":("1000002","7892840800246"), "CINI":("999999","7896188403077")}
r = raw[raw["preco_regular"]>0]
for marca,(ph,ean) in pares.items():
    a = r[(r["id_produto"]==ph) & (r["__mes"].astype(str)=="2024-09")]
    e = r[(r["id_produto"]==ean) & (r["__mes"].astype(str)=="2024-10")]
    la = a.groupby("endereco_completo")["preco_regular"].median()
    le = e.groupby("endereco_completo")["preco_regular"].median()
    comuns = la.index.intersection(le.index)
    var = (le[comuns] / la[comuns] - 1) * 100
    print(marca, "| lojas set/24:", len(la), "| lojas out/24:", len(le), "| em comum:", len(comuns),
          "| variacao mediana nas lojas comuns (%):", round(var.median(), 1) if len(comuns) else None,
          "| min/max:", (round(var.min(),1), round(var.max(),1)) if len(comuns) else None)

In [ ]:
r = raw[raw["preco_regular"]>0]
a = r[(r["id_produto"]=="999999") & (r["__mes"].astype(str)=="2024-09")].groupby("endereco_completo")["preco_regular"].median()
e = r[(r["id_produto"]=="7896188403077") & (r["__mes"].astype(str)=="2024-10")].groupby("endereco_completo")["preco_regular"].median()
t = pd.DataFrame({"set24": a, "out24": e}).dropna()
t["var_pct"] = ((t["out24"] / t["set24"] - 1) * 100).round(1)
print(t.sort_values("var_pct").to_string())

In [ ]:
corr = pd.DataFrame([
    ["1000000","REFRIGERANTE COCA COLA Pet -2 LITRO","7894900027013","candidato","sucessao_temporal_sem_sobreposicao","marca, volume e embalagem; 0 meses em comum; mediana nas 10 lojas comuns set->out/24: 0,0%","pendente"],
    ["1000001","REFRIGERANTE GUARANA ANTARTICA Pet -2 LITRO","7891991001342","candidato","sucessao_temporal_sem_sobreposicao","marca, volume e embalagem; 0 meses em comum; mediana nas 11 lojas comuns set->out/24: 0,0%","pendente"],
    ["1000002","REFRIGERANTE PEPSI COLA Pet -2 LITRO","7892840800246","candidato","sucessao_temporal_sem_sobreposicao","marca, volume e embalagem; 0 meses em comum; mediana nas 7 lojas comuns set->out/24: 0,0%","pendente"],
    ["999999","REFRIGERANTE CINI Pet -2 LITRO","7896188403077","candidato","sucessao_temporal_sem_sobreposicao","marca, volume e embalagem; 0 meses em comum; mediana nas 9 lojas comuns set->out/24: -3,1% (2 quedas grandes, sem altas)","pendente; ressalva Cini"],
    ["7896050600269","FEIJAO PRETO -RESERVA -1 KG","7898171400209","candidato","texto_exato_normalizado","descricao normalizada identica; 75 linhas; mediana mensal igual em 2 de 9 meses comuns e ate 6% acima nos demais com n razoavel; mix de lojas nao testado","pendente"],
], columns=["codigo_origem","descricao_origem","sku_canonico","tipo_vinculo","metodo","evidencia","status"])
corr["data"] = "2026-11-07"
corr["revisor"] = ""
corr.to_csv("raw_data/correspondencia_rascunho.csv", index=False)
print(corr.shape)
print(open("raw_data/correspondencia_rascunho.csv").readline())

In [ ]:
chaves = ["id_produto","id_empresa","endereco_completo","data_pesquisa"]
ext = raw[raw["id_produto"].isin(corr["codigo_origem"]) & (raw["preco_regular"] > 0)].copy()
ext = ext[~ext["descricao"].str.contains("BARATO", na=False)]
ext["sku"] = ext["id_produto"].map(dict(zip(corr["codigo_origem"], corr["sku_canonico"])))
ext = ext.sort_values("__mes").drop_duplicates(chaves, keep="first")
ext["__mes"] = ext["__mes"].astype(str)
print("linhas candidatas (positivas, unicas):", len(ext))
print(ext.groupby("sku").size().to_string())

base = obs[obs["id_produto"].isin(corr["sku_canonico"])].copy()
base["sku"] = base["id_produto"]
ganho = pd.DataFrame({
    "meses_antes": base.groupby("sku")["__mes"].nunique(),
    "registros_antes": base.groupby("sku").size(),
    "meses_depois": pd.concat([base[["sku","__mes"]], ext[["sku","__mes"]]]).groupby("sku")["__mes"].nunique(),
    "registros_candidatos": ext.groupby("sku").size(),
})
ganho["registros_depois"] = ganho["registros_antes"] + ganho["registros_candidatos"]
print(ganho.to_string())
print("total registros antes:", ganho["registros_antes"].sum(), "| depois:", ganho["registros_depois"].sum())

In [ ]:
ganho_out = ganho.reset_index().rename(columns={"sku": "id_produto"})
ganho_out = cat.merge(ganho_out, on="id_produto", how="inner")
ganho_out.to_csv("raw_data/ganho_candidatos.csv", index=False)
print(ganho_out.shape)
print(ganho_out[["grupo","descricao","meses_antes","meses_depois","registros_antes","registros_depois"]].to_string(index=False))

In [ ]:
import pandas as pd
c13 = pd.read_csv("dados/catalogo_expansao_13.csv", dtype=str)
print("grupos no arquivo:", c13["grupo"].unique().tolist())

ids = c13["id_produto"].tolist()
x = raw[raw["id_produto"].isin(ids)]
moda = x.groupby("id_produto")["codigo_categoria"].agg(lambda s: s.dropna().astype(str).mode().iloc[0] if s.notna().any() else None)
nuniq = x.groupby("id_produto")["codigo_categoria"].nunique()

t = c13[["grupo","id_produto","descricao"]].copy()
t["codigo_categoria"] = t["id_produto"].map(moda)
t["n_codigos_distintos"] = t["id_produto"].map(nuniq)
g = t["grupo"].str.lower()
t["tipo"] = g.map(lambda s: "integral" if "leite" in s else ("soja" if "leo" in s else "pilsen"))
t["embalagem"] = g.map(lambda s: "1 L" if "leite" in s else ("900 ml" if "leo" in s else "350 ml"))
t = t[["grupo","codigo_categoria","tipo","embalagem","id_produto","descricao","n_codigos_distintos"]]
print(t.to_csv(sep="\t", index=False))

In [ ]:
import re, unicodedata
import pandas as pd
from mercado.base_precos import carregar_base, carregar_catalogo, limpar, CHAVE_DUP

raw = carregar_base()
cat = carregar_catalogo()
obs = limpar(raw, cat)
print("obs:", len(obs))

b = raw[raw["preco_regular"] > 0].sort_values("__mes").drop_duplicates(CHAVE_DUP, keep="first").copy()
print("base positiva unica:", len(b), "de", len(raw))

def norm(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", s.upper()).strip()
b["dn"] = b["descricao"].map(norm)

def ean_ok(c):
    c = str(c)
    if not (c.isdigit() and len(c) == 13):
        return False
    s = sum(int(d) * (3 if i % 2 else 1) for i, d in enumerate(c[:12]))
    return (10 - s % 10) % 10 == int(c[12])
b["ean_ok"] = b["id_produto"].map(ean_ok)
b["generico"] = b["dn"].str.contains("BARATO", na=False)
b["no_cat"] = b["id_produto"].isin(cat["id_produto"])

grupos = {"Arroz": r"^ARROZ\b", "Feijao": r"^FEIJAO\b", "Macarrao": r"^MACARR?AO\b", "Acucar": r"^ACUCAR\b",
          "Cafe": r"^CAFE\b", "Refrigerante": r"^REFRIGERANTE\b", "Creme dental": r"^CREME DENTAL\b",
          "Sabonete": r"^SABONETE\b", "Agua mineral": r"^AGUA MINERAL\b", "Leite": r"^LEITE\b",
          "Oleo de soja": r"^OLEO DE SOJA\b", "Cerveja": r"^CERVEJA\b"}
b["grupo_txt"] = None
for g, rx in grupos.items():
    m = b["grupo_txt"].isna() & b["dn"].str.contains(rx, regex=True, na=False)
    b.loc[m, "grupo_txt"] = g

u = b[b["grupo_txt"].notna()]
fora = u[u["ean_ok"] & ~u["no_cat"] & ~u["generico"]]
t = u.groupby("grupo_txt").agg(
    linhas=("dn", "size"),
    ean_valido=("ean_ok", "sum"),
    genericas=("generico", "sum"),
    no_catalogo=("no_cat", "sum"),
    ids=("id_produto", "nunique"),
)
t["ean_valido_fora_cat"] = fora.groupby("grupo_txt").size()
t["ids_ean_fora_cat"] = fora.groupby("grupo_txt")["id_produto"].nunique()
t = t.fillna(0).astype(int)
t.loc["TOTAL"] = t.sum()
print(t.to_string())
print("catalogo capturado pelo regex:", int(u["no_cat"].sum()), "| obs do catalogo:", len(obs))
print(b.groupby("codigo_categoria", dropna=False).agg(linhas=("dn", "size"), dos_12_grupos=("grupo_txt", lambda s: s.notna().sum())).to_string())

In [ ]:
print(pd.crosstab(u["grupo_txt"], u["codigo_categoria"].fillna("NaN"), margins=True).to_string())
resto = u[~u["ean_ok"] & ~u["generico"]]
print("linhas sem EAN valido e nao genericas:", len(resto))
print(resto.groupby("grupo_txt").agg(linhas=("dn", "size"), ids=("id_produto", "nunique")).to_string())
print(resto.groupby(["grupo_txt", "id_produto", "dn"]).size().sort_values(ascending=False).head(15).to_string())

In [ ]:
pares = {"SKOL":("999997","7891149200504"), "ANTARTICA":("999991","7891991000796"),
         "KAISER":("999995","7896045512317"), "BRAHMA":("999993","7891149010509")}
r = raw[raw["preco_regular"]>0]
for marca,(ph,ean) in pares.items():
    a = r[r["id_produto"]==ph]; e = r[r["id_produto"]==ean]
    am = sorted(set(a["__mes"].astype(str))); em = sorted(set(e["__mes"].astype(str)))
    comuns = sorted(set(am) & set(em))
    ult_a = a[a["__mes"].astype(str)==am[-1]]["endereco_completo"].unique() if am else []
    print(marca, "| placeholder:", am[0], "a", am[-1], "n=", len(a), "meses=", len(am), "med=", a["preco_regular"].median(),
          "| EAN:", em[0], "a", em[-1], "n=", len(e), "meses=", len(em), "med=", e["preco_regular"].median(),
          "| meses em comum:", comuns)

In [ ]:
pares = {"SKOL":("999997","7891149200504"), "ANTARTICA":("999991","7891991000796"),
         "KAISER":("999995","7896045512317"), "BRAHMA":("999993","7891149010509")}
r = raw[raw["preco_regular"]>0]
for marca,(ph,ean) in pares.items():
    a = r[(r["id_produto"]==ph) & (r["__mes"].astype(str)=="2024-09")]
    e = r[(r["id_produto"]==ean) & (r["__mes"].astype(str)=="2024-10")]
    la = a.groupby("endereco_completo")["preco_regular"].median()
    le = e.groupby("endereco_completo")["preco_regular"].median()
    comuns = la.index.intersection(le.index)
    var = (le[comuns] / la[comuns] - 1) * 100
    print(marca, "| lojas set/24:", len(la), "| lojas out/24:", len(le), "| em comum:", len(comuns),
          "| variacao mediana (%):", round(var.median(), 1) if len(comuns) else None,
          "| min/max:", (round(var.min(),1), round(var.max(),1)) if len(comuns) else None)

In [ ]:
novos = pd.DataFrame([
    ["999997","CERVEJA SKOL LATA -350 ML","7891149200504"],
    ["999991","CERVEJA ANTARTICA LATA -350 ML","7891991000796"],
    ["999995","CERVEJA KAISER LATA -350 ML","7896045512317"],
    ["999993","CERVEJA BRAHMA LATA -350 ML","7891149010509"]],
    columns=["codigo_origem","descricao_origem","sku_canonico"])
novos["tipo_vinculo"] = "candidato"
novos["metodo"] = "sucessao_temporal_sem_sobreposicao"
novos["evidencia"] = "marca, volume e embalagem; 0 meses em comum; mediana nas lojas comuns set->out/24: 0,0% (8 a 10 lojas)"
novos["status"] = "pendente"; novos["data"] = "2026-11-07"; novos["revisor"] = ""
corr2 = pd.concat([corr, novos], ignore_index=True)
corr2.to_csv("raw_data/correspondencia_rascunho.csv", index=False)
print(corr2.shape)

chaves = ["id_produto","id_empresa","endereco_completo","data_pesquisa"]
ext = raw[raw["id_produto"].isin(novos["codigo_origem"]) & (raw["preco_regular"] > 0)].copy()
ext["sku"] = ext["id_produto"].map(dict(zip(novos["codigo_origem"], novos["sku_canonico"])))
ext = ext.sort_values("__mes").drop_duplicates(chaves, keep="first")
ext["__mes"] = ext["__mes"].astype(str)
base = obs[obs["id_produto"].isin(novos["sku_canonico"])].copy()
base["sku"] = base["id_produto"]; base["__mes"] = base["__mes"].astype(str)
g = pd.DataFrame({
    "meses_antes": base.groupby("sku")["__mes"].nunique(),
    "registros_antes": base.groupby("sku").size(),
    "meses_depois": pd.concat([base[["sku","__mes"]], ext[["sku","__mes"]]]).groupby("sku")["__mes"].nunique(),
    "registros_candidatos": ext.groupby("sku").size()})
g["registros_depois"] = g["registros_antes"] + g["registros_candidatos"]
print(g.to_string())
print("candidatos cerveja:", g["registros_candidatos"].sum(), "| antes:", g["registros_antes"].sum(), "| depois:", g["registros_depois"].sum())

In [ ]:
corr = pd.read_csv("raw_data/correspondencia_rascunho.csv", dtype=str).fillna("")
print("corr lido:", corr.shape)

novos = pd.DataFrame([
    ["999997","CERVEJA SKOL LATA -350 ML","7891149200504"],
    ["999991","CERVEJA ANTARTICA LATA -350 ML","7891991000796"],
    ["999995","CERVEJA KAISER LATA -350 ML","7896045512317"],
    ["999993","CERVEJA BRAHMA LATA -350 ML","7891149010509"]],
    columns=["codigo_origem","descricao_origem","sku_canonico"])
novos["tipo_vinculo"] = "candidato"
novos["metodo"] = "sucessao_temporal_sem_sobreposicao"
novos["evidencia"] = "marca, volume e embalagem; 0 meses em comum; mediana nas lojas comuns set->out/24: 0,0% (8 a 10 lojas)"
novos["status"] = "pendente"; novos["data"] = "2026-11-07"; novos["revisor"] = ""
corr2 = pd.concat([corr, novos], ignore_index=True)
corr2.to_csv("raw_data/correspondencia_rascunho.csv", index=False)
print(corr2.shape)

chaves = ["id_produto","id_empresa","endereco_completo","data_pesquisa"]
ext = raw[raw["id_produto"].isin(novos["codigo_origem"]) & (raw["preco_regular"] > 0)].copy()
ext["sku"] = ext["id_produto"].map(dict(zip(novos["codigo_origem"], novos["sku_canonico"])))
ext = ext.sort_values("__mes").drop_duplicates(chaves, keep="first")
ext["__mes"] = ext["__mes"].astype(str)
base = obs[obs["id_produto"].isin(novos["sku_canonico"])].copy()
base["sku"] = base["id_produto"]; base["__mes"] = base["__mes"].astype(str)
g = pd.DataFrame({
    "meses_antes": base.groupby("sku")["__mes"].nunique(),
    "registros_antes": base.groupby("sku").size(),
    "meses_depois": pd.concat([base[["sku","__mes"]], ext[["sku","__mes"]]]).groupby("sku")["__mes"].nunique(),
    "registros_candidatos": ext.groupby("sku").size()})
g["registros_depois"] = g["registros_antes"] + g["registros_candidatos"]
print(g.to_string())
print("candidatos cerveja:", g["registros_candidatos"].sum(), "| antes:", g["registros_antes"].sum(), "| depois:", g["registros_depois"].sum())

In [ ]:
m = fora[fora["grupo_txt"]=="Macarrao"]
t = m.groupby(["id_produto","dn"]).agg(linhas=("dn","size"), meses=("__mes","nunique"), ini=("__mes","min"), fim=("__mes","max")).reset_index().sort_values("linhas", ascending=False)
print(len(t), "descricoes em", t["id_produto"].nunique(), "codigos |", t["linhas"].sum(), "linhas")
print(t.head(25).to_string(index=False))

In [ ]:
corr = pd.read_csv("raw_data/correspondencia_rascunho.csv", dtype=str).fillna("")
print("candidatos na tabela:", len(corr), "| codigos unicos:", corr["codigo_origem"].nunique())
mapa = dict(zip(corr["codigo_origem"], corr["sku_canonico"]))

cand = raw[raw["id_produto"].isin(mapa) & (raw["preco_regular"] > 0)].copy()
cand = cand.sort_values("__mes").drop_duplicates(CHAVE_DUP, keep="first")
cand["id_origem"] = cand["id_produto"]
cand["id_produto"] = cand["id_origem"].map(mapa)
cand["camada"] = "candidato"
o = obs.copy()
o["id_origem"] = o["id_produto"]
o["camada"] = "observado"

cols = ["id_produto", "id_origem", "camada", "__mes", "data_pesquisa", "rede", "endereco_completo", "preco_regular"]
todas = pd.concat([o[cols], cand[cols]], ignore_index=True)
todas["__mes"] = todas["__mes"].astype(str)
print("sobreposicao entre camadas (mesmo SKU, loja e data):", todas.duplicated(["id_produto", "endereco_completo", "data_pesquisa"]).sum())

def resumo(df, suf):
    return df.groupby("id_produto").agg(**{
        f"registros_{suf}": ("preco_regular", "size"),
        f"meses_{suf}": ("__mes", "nunique"),
        f"primeira_{suf}": ("__mes", "min"),
        f"ultima_{suf}": ("__mes", "max")})

ob = todas[todas["camada"] == "observado"]
lojas = ob.groupby("id_produto").agg(redes_obs=("rede", "nunique"), lojas_obs=("endereco_completo", "nunique"))
cp = (cat[["grupo", "id_produto", "descricao"]].set_index("id_produto")
      .join(resumo(ob, "obs")).join(lojas).join(resumo(todas, "total")).reset_index())
cp["registros_cand"] = cp["registros_total"] - cp["registros_obs"]
cp["meses_ganhos"] = cp["meses_total"] - cp["meses_obs"]
cp.to_csv("raw_data/cobertura_por_camada.csv", index=False)
print(cp.shape)
print("observado:", cp["registros_obs"].sum(), "| candidato:", cp["registros_cand"].sum(), "| total:", cp["registros_total"].sum())
print(cp[cp["registros_cand"] > 0][["grupo", "descricao", "registros_obs", "registros_cand", "meses_obs", "meses_total", "primeira_total", "ultima_total"]].to_string(index=False))

In [ ]:
k = ["id_produto", "endereco_completo", "data_pesquisa"]
d = todas[todas.duplicated(k, keep=False)].copy()
print("linhas envolvidas:", len(d), "| chaves distintas:", d.groupby(k).ngroups)
print(d.groupby(["id_produto", "id_origem", "camada"]).size().to_string())
x = d.groupby(k).agg(n=("preco_regular", "size"), precos=("preco_regular", "nunique"), pmin=("preco_regular", "min"), pmax=("preco_regular", "max")).reset_index()
print("pares com preco igual:", (x["precos"] == 1).sum(), "| com preco diferente:", (x["precos"] > 1).sum())
print(x.sort_values("pmax", ascending=False).head(10).to_string(index=False))

In [ ]:
todas = todas.sort_values("camada", key=lambda s: s.map({"observado": 0, "candidato": 1}))
todas = todas.drop_duplicates(["id_produto", "endereco_completo", "data_pesquisa"], keep="first")
print("sobreposicao restante:", todas.duplicated(["id_produto", "endereco_completo", "data_pesquisa"]).sum())

ob = todas[todas["camada"] == "observado"]
lojas = ob.groupby("id_produto").agg(redes_obs=("rede", "nunique"), lojas_obs=("endereco_completo", "nunique"))
cp = (cat[["grupo", "id_produto", "descricao"]].set_index("id_produto")
      .join(resumo(ob, "obs")).join(lojas).join(resumo(todas, "total")).reset_index())
cp["registros_cand"] = cp["registros_total"] - cp["registros_obs"]
cp["meses_ganhos"] = cp["meses_total"] - cp["meses_obs"]
cp.to_csv("raw_data/cobertura_por_camada.csv", index=False)
print("observado:", cp["registros_obs"].sum(), "| candidato:", cp["registros_cand"].sum(), "| total:", cp["registros_total"].sum())
print(cp[cp["registros_cand"] > 0][["descricao", "registros_obs", "registros_cand", "meses_obs", "meses_total"]].to_string(index=False))

corr["evidencia"] = corr.apply(lambda r: r["evidencia"] + "; 39 linhas coincidem em loja e data com o EAN do catalogo (38 com preco igual): removidas; vinculo com ressalva" if r["codigo_origem"] == "7896050600269" else r["evidencia"], axis=1)
corr.to_csv("raw_data/correspondencia_rascunho.csv", index=False)
print(corr[corr["codigo_origem"] == "7896050600269"]["evidencia"].iloc[0])

In [ ]:
def num(s):
    return pd.to_numeric(s.astype(str).str.replace(",", ".", regex=False), errors="coerce")

ids = set(cat["id_produto"]) | set(corr["codigo_origem"])
z = raw[raw["id_produto"].isin(ids) & ~(raw["preco_regular"] > 0)].copy()
print("linhas com preco regular zerado/vazio:", len(z), "| em codigos do catalogo:", z["id_produto"].isin(cat["id_produto"]).sum(), "| em codigos candidatos:", z["id_produto"].isin(corr["codigo_origem"]).sum())

for c in ["preco_promocao", "preco_fidelidade", "preco_atacado"]:
    z[c + "_n"] = num(z[c])
z["alguma"] = (z[["preco_promocao_n", "preco_fidelidade_n", "preco_atacado_n"]] > 0).any(axis=1)
print("com promocao > 0:", (z["preco_promocao_n"] > 0).sum(), "| fidelidade > 0:", (z["preco_fidelidade_n"] > 0).sum(), "| atacado > 0:", (z["preco_atacado_n"] > 0).sum(), "| qualquer uma:", z["alguma"].sum())

pos_keys = raw[raw["id_produto"].isin(ids) & (raw["preco_regular"] > 0)][CHAVE_DUP].drop_duplicates()
zz = z[z["alguma"]].merge(pos_keys.assign(tem_regular=True), on=CHAVE_DUP, how="left")
print("das que tem outro preco, ja tem linha regular na mesma loja e data:", zz["tem_regular"].fillna(False).sum(), "| sem:", (~zz["tem_regular"].fillna(False).astype(bool)).sum())
print(zz.groupby(["id_produto"]).size().sort_values(ascending=False).head(10).to_string())
print(zz[["id_produto", "__mes", "preco_promocao_n", "preco_fidelidade_n", "preco_atacado_n"]].head(8).to_string(index=False))

In [ ]:
import numpy as np
c = obs[obs["id_produto"]=="7894900027013"].copy()
c["__mes"] = c["__mes"].astype(str)
ult = sorted(c["__mes"].unique())[-3:]
w = c[c["__mes"].isin(ult)]
loja = w.groupby("endereco_completo")["preco_regular"].median()
rede = w.groupby("rede")["preco_regular"].median()
rng = np.random.default_rng(0)
boot = [np.median(rng.choice(loja.values, len(loja), replace=True)) for _ in range(2000)]
print("janela:", ult, "| obs:", len(w), "| lojas:", len(loja), "| redes:", len(rede))
print("mediana lojas:", loja.median(), "| IC95%:", np.percentile(boot, [2.5, 97.5]).round(2))
print("mediana entre redes:", rede.median(), "| P25-P75 lojas:", loja.quantile([.25, .75]).round(2).tolist())
print(rede.sort_values().to_string())

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
o = obs.copy(); o["__mes"] = o["__mes"].astype(str)
ult = sorted(o["__mes"].unique())[-3:]
linhas = []
for e, g in o.groupby("id_produto"):
    ms = sorted(g["__mes"].unique())[-3:]
    w = g[g["__mes"].isin(ms)]
    loja = w.groupby("endereco_completo")["preco_regular"].median()
    med = loja.median()
    ic = np.percentile([np.median(rng.choice(loja.values, len(loja), replace=True)) for _ in range(1000)], [2.5, 97.5])
    linhas.append({"id_produto": e, "ultima_janela": ms[-1], "lojas": len(loja), "redes": w["rede"].nunique(),
                   "mediana": med, "ic_largura_pct": (ic[1]-ic[0])/med*100})
r = pd.DataFrame(linhas).merge(cat[["id_produto","grupo","descricao"]], on="id_produto")
print("janela mais recente da base:", ult[-1])
print(r["ic_largura_pct"].describe().round(1).to_string())
print("SKUs com IC > 10% da mediana:", (r["ic_largura_pct"] > 10).sum(), "de", len(r))
print(r.sort_values("ic_largura_pct", ascending=False).head(12)[["grupo","descricao","ultima_janela","lojas","redes","mediana","ic_largura_pct"]].round(2).to_string(index=False))

In [ ]:
rng = np.random.default_rng(0)
def ic_larg(g, n):
    ms = sorted(g["__mes"].unique())[-n:]
    loja = g[g["__mes"].isin(ms)].groupby("endereco_completo")["preco_regular"].median()
    ic = np.percentile([np.median(rng.choice(loja.values, len(loja), replace=True)) for _ in range(1000)], [2.5, 97.5])
    return (ic[1]-ic[0])/loja.median()*100, len(loja)
rows = []
for e, g in o.groupby("id_produto"):
    l3, n3 = ic_larg(g, 3); l6, n6 = ic_larg(g, 6)
    rows.append({"id_produto": e, "ic3": l3, "lojas3": n3, "ic6": l6, "lojas6": n6})
q = pd.DataFrame(rows).merge(cat[["id_produto","grupo","descricao"]], on="id_produto")
print("IC>10% com 3 meses:", (q["ic3"]>10).sum(), "| com 6 meses:", (q["ic6"]>10).sum())
print(q.sort_values("ic6", ascending=False).head(10)[["descricao","lojas3","ic3","lojas6","ic6"]].round(1).to_string(index=False))

In [ ]:
q["regra_janela"] = np.where(q["ic3"] <= 10, "3 meses", np.where(q["ic6"] <= 10, "6 meses", "baixa precisao"))
q = q[["grupo","id_produto","descricao","lojas3","ic3","lojas6","ic6","regra_janela"]].round(1)
q.to_csv("raw_data/precisao_por_sku.csv", index=False)
print(q["regra_janela"].value_counts().to_string())
print(q.shape)

In [ ]:
import zlib
def ic_larg2(g, n):
    ms = sorted(g["__mes"].unique())[-n:]
    loja = g[g["__mes"].isin(ms)].groupby("endereco_completo")["preco_regular"].median()
    rng = np.random.default_rng(zlib.crc32(str(g["id_produto"].iloc[0]).encode()))
    ic = np.percentile([np.median(rng.choice(loja.values, len(loja), replace=True)) for _ in range(1000)], [2.5, 97.5])
    return (ic[1]-ic[0])/loja.median()*100, len(loja), loja.median(), ms[-1]
rows = []
for e, g in o.groupby("id_produto"):
    l3, n3, m3, f3 = ic_larg2(g, 3); l6, n6, m6, f6 = ic_larg2(g, 6)
    rows.append({"id_produto": e, "mediana3": m3, "janela_fim": f3, "lojas3": n3, "ic3": l3, "lojas6": n6, "ic6": l6})
q2 = pd.DataFrame(rows).merge(cat[["id_produto","grupo","descricao"]], on="id_produto")
q2["regra_janela"] = np.where(q2["ic3"] <= 10, "3 meses", np.where(q2["ic6"] <= 10, "6 meses", "baixa precisao"))
q2["referencia_antiga"] = q2["janela_fim"] < "2026-10"
q2 = q2[["grupo","id_produto","descricao","mediana3","janela_fim","lojas3","ic3","lojas6","ic6","regra_janela","referencia_antiga"]].round(2)
q2.to_csv("raw_data/precisao_por_sku.csv", index=False)
print(q2["regra_janela"].value_counts().to_string())
print("referencia antiga:", q2["referencia_antiga"].sum(), "| shape:", q2.shape)

In [ ]:
rows = []
for e, g in o.groupby("id_produto"):
    l3, n3, m3, f3 = ic_larg2(g, 3); l6, n6, m6, f6 = ic_larg2(g, 6)
    rows.append({"id_produto": e, "ic3": round(l3, 2), "ic6": round(l6, 2)})
chk = pd.DataFrame(rows)
sal = pd.read_csv("raw_data/precisao_por_sku.csv", dtype={"id_produto": str})
m = chk.merge(sal[["id_produto", "ic3", "ic6"]], on="id_produto", suffixes=("_novo", "_salvo"))
print("SKUs:", len(m), "| ic3 diferente:", (m["ic3_novo"] != m["ic3_salvo"]).sum(), "| ic6 diferente:", (m["ic6_novo"] != m["ic6_salvo"]).sum())

In [ ]:
import numpy as np
c = obs[obs["id_produto"]=="7894900027013"].copy()
c["__mes"] = c["__mes"].astype(str)
meses = sorted(c["__mes"].unique())
rows = []
for i in range(3, len(meses)):
    m = meses[i]
    alvo = c[c["__mes"]==m].groupby("endereco_completo")["preco_regular"].median()
    ref3 = c[c["__mes"].isin(meses[i-3:i])].groupby("endereco_completo")["preco_regular"].median().median()
    ref1 = c[c["__mes"]==meses[i-1]].groupby("endereco_completo")["preco_regular"].median().median()
    rows.append({"mes": m, "lojas_alvo": len(alvo), "ref3": ref3, "ref1": ref1,
                 "mae3": (alvo - ref3).abs().mean(), "mae1": (alvo - ref1).abs().mean(),
                 "med_alvo": alvo.median(), "erro_med3": ref3 - alvo.median(), "erro_med1": ref1 - alvo.median()})
t = pd.DataFrame(rows)
print(t.round(2).to_string(index=False))
print("meses testados:", len(t))
print("MAE medio por loja (R$) | janela 3m:", round(t["mae3"].mean(), 2), "| ultimo mes:", round(t["mae1"].mean(), 2))
print("erro absoluto da mediana (R$) | janela 3m:", round(t["erro_med3"].abs().mean(), 2), "| ultimo mes:", round(t["erro_med1"].abs().mean(), 2))
print("janela 3m melhor que ultimo mes em", (t["mae3"] < t["mae1"]).sum(), "de", len(t), "meses")

In [ ]:
o2 = obs.copy(); o2["__mes"] = o2["__mes"].astype(str)
res = []
for e, g in o2.groupby("id_produto"):
    ms = sorted(g["__mes"].unique())
    a3, a1, n = [], [], 0
    for i in range(3, len(ms)):
        alvo = g[g["__mes"]==ms[i]].groupby("endereco_completo")["preco_regular"].median()
        r3 = g[g["__mes"].isin(ms[i-3:i])].groupby("endereco_completo")["preco_regular"].median().median()
        r1 = g[g["__mes"]==ms[i-1]].groupby("endereco_completo")["preco_regular"].median().median()
        a3.append(abs(alvo.median()-r3)); a1.append(abs(alvo.median()-r1)); n += 1
    if n:
        res.append({"id_produto": e, "meses_testados": n, "err3": np.mean(a3), "err1": np.mean(a1)})
R = pd.DataFrame(res).merge(cat[["id_produto","grupo","descricao"]], on="id_produto")
R["melhor"] = np.where(R["err3"] < R["err1"], "3 meses", np.where(R["err3"] > R["err1"], "ultimo mes", "empate"))
print(R["melhor"].value_counts().to_string())
print("erro medio da mediana (R$) | 3m:", round(R["err3"].mean(), 3), "| ultimo mes:", round(R["err1"].mean(), 3))
print(R.sort_values("meses_testados").head(5)[["descricao","meses_testados","err3","err1"]].round(2).to_string(index=False))

In [ ]:
R["err3_pct"] = R["err3"] / R["id_produto"].map(o2.groupby("id_produto")["preco_regular"].median()) * 100
R["err1_pct"] = R["err1"] / R["id_produto"].map(o2.groupby("id_produto")["preco_regular"].median()) * 100
print("erro medio em % do preco | 3m:", round(R["err3_pct"].mean(), 2), "| ultimo mes:", round(R["err1_pct"].mean(), 2))
print("mediana do erro em % | 3m:", round(R["err3_pct"].median(), 2), "| ultimo mes:", round(R["err1_pct"].median(), 2))
print(R.groupby("grupo")[["err3_pct", "err1_pct"]].mean().round(2).to_string())
R.round(3).to_csv("raw_data/mae_por_sku.csv", index=False)
print(R.shape)

In [ ]:
def n2(s):
    s = norm(s)
    s = re.sub(r"\bC/\s*", "COM ", s)
    s = re.sub(r"\bPET\b", " ", s)
    s = re.sub(r"\bA VACUO\b", "VACUO", s)
    s = re.sub(r"(\d)\s*(KG|GR|G|ML|L)\b", r"\1 \2", s)
    s = s.replace("-", " ")
    return re.sub(r"\s+", " ", s).strip()
catn = cat.assign(k=cat["descricao"].map(n2))
f = fora.assign(k=fora["descricao"].map(n2))
hit = f.merge(catn[["id_produto","k","descricao"]].rename(columns={"id_produto":"sku","descricao":"desc_cat"}), on="k")
print("codigos fora do catalogo com descricao igual a um SKU:", hit["id_produto"].nunique(), "| linhas:", len(hit))
print(hit.groupby(["id_produto","sku","descricao","desc_cat"]).agg(linhas=("k","size"), ini=("__mes","min"), fim=("__mes","max")).to_string())

In [ ]:
import zlib
import numpy as np
import pandas as pd
from mercado.base_precos import carregar_base, carregar_catalogo, limpar, CHAVE_DUP

raw = carregar_base(); cat = carregar_catalogo(); obs = limpar(raw, cat)
obs["__mes"] = obs["__mes"].astype(str)
print("obs:", len(obs))

def salvar(df, nome):
    for p in ["raw_data", "dados"]:
        df.to_csv(f"{p}/{nome}", index=False)

# 1. correspondencia: Feijao Reserva vira conflito
corr = pd.read_csv("raw_data/correspondencia_rascunho.csv", dtype=str).fillna("")
m = corr["codigo_origem"] == "7896050600269"
corr.loc[m, "tipo_vinculo"] = "conflito"
corr.loc[m, "status"] = "excluido"
if not corr.loc[m, "evidencia"].str.contains("Cosmos").any():
    corr.loc[m, "evidencia"] += "; SI5: nao encontrado; Cosmos (nao auditado): FEIJAO CARIOCA/PRETO 1KG, marca Reserva; prefixo 7896050 diferente do fabricante (7898171)"
salvar(corr, "correspondencia_rascunho.csv")
print(corr[["codigo_origem", "sku_canonico", "tipo_vinculo", "status"]].to_string(index=False))

# 2. cobertura por SKU
antes = raw[raw["id_produto"].isin(cat["id_produto"]) & (raw["preco_regular"] > 0)]
cob = obs.groupby("id_produto").agg(registros=("preco_regular", "size"), meses=("__mes", "nunique"), redes=("rede", "nunique"),
                                    lojas=("endereco_completo", "nunique"), primeira=("__mes", "min"), ultima=("__mes", "max")).reset_index()
cob = cat.merge(cob, on="id_produto", how="left")
cob["registros_antes"] = cob["id_produto"].map(antes.groupby("id_produto").size()).fillna(0).astype(int)
cob["removidos"] = cob["registros_antes"] - cob["registros"]
salvar(cob, "cobertura_por_sku.csv")
print("cobertura: antes", cob["registros_antes"].sum(), "| depois", cob["registros"].sum())

# 3. cobertura por camada (so candidatos aceitos)
ok = corr[corr["tipo_vinculo"] == "candidato"]
mapa = dict(zip(ok["codigo_origem"], ok["sku_canonico"]))
cand = raw[raw["id_produto"].isin(mapa) & (raw["preco_regular"] > 0)].copy()
cand = cand.sort_values("__mes").drop_duplicates(CHAVE_DUP, keep="first")
cand["id_origem"] = cand["id_produto"]; cand["id_produto"] = cand["id_origem"].map(mapa); cand["camada"] = "candidato"
o = obs.copy(); o["id_origem"] = o["id_produto"]; o["camada"] = "observado"
cols = ["id_produto", "id_origem", "camada", "__mes", "data_pesquisa", "rede", "endereco_completo", "preco_regular"]
todas = pd.concat([o[cols], cand[cols]], ignore_index=True)
todas["__mes"] = todas["__mes"].astype(str)
k = ["id_produto", "endereco_completo", "data_pesquisa"]
print("sobreposicao entre camadas:", todas.duplicated(k).sum())
todas = todas.sort_values("camada", key=lambda s: s.map({"observado": 0, "candidato": 1})).drop_duplicates(k, keep="first")
def resumo(df, suf):
    return df.groupby("id_produto").agg(**{f"registros_{suf}": ("preco_regular", "size"), f"meses_{suf}": ("__mes", "nunique"),
                                           f"primeira_{suf}": ("__mes", "min"), f"ultima_{suf}": ("__mes", "max")})
ob = todas[todas["camada"] == "observado"]
lojas = ob.groupby("id_produto").agg(redes_obs=("rede", "nunique"), lojas_obs=("endereco_completo", "nunique"))
cp = cat[["grupo", "id_produto", "descricao"]].set_index("id_produto").join(resumo(ob, "obs")).join(lojas).join(resumo(todas, "total")).reset_index()
cp["registros_cand"] = cp["registros_total"] - cp["registros_obs"]
cp["meses_ganhos"] = cp["meses_total"] - cp["meses_obs"]
salvar(cp, "cobertura_por_camada.csv")
print("camadas: observado", cp["registros_obs"].sum(), "| candidato", cp["registros_cand"].sum(), "| total", cp["registros_total"].sum())

# 4. precisao da referencia (semente fixa por SKU)
def ic_larg2(g, n):
    ms = sorted(g["__mes"].unique())[-n:]
    loja = g[g["__mes"].isin(ms)].groupby("endereco_completo")["preco_regular"].median()
    rng = np.random.default_rng(zlib.crc32(str(g["id_produto"].iloc[0]).encode()))
    ic = np.percentile([np.median(rng.choice(loja.values, len(loja), replace=True)) for _ in range(1000)], [2.5, 97.5])
    return (ic[1] - ic[0]) / loja.median() * 100, len(loja), loja.median(), ms[-1]
rows = []
for e, g in obs.groupby("id_produto"):
    l3, n3, m3, f3 = ic_larg2(g, 3); l6, n6, m6, f6 = ic_larg2(g, 6)
    rows.append({"id_produto": e, "mediana3": m3, "janela_fim": f3, "lojas3": n3, "ic3": l3, "lojas6": n6, "ic6": l6})
q2 = pd.DataFrame(rows).merge(cat[["id_produto", "grupo", "descricao"]], on="id_produto")
q2["regra_janela"] = np.where(q2["ic3"] <= 10, "3 meses", np.where(q2["ic6"] <= 10, "6 meses", "baixa precisao"))
q2["referencia_antiga"] = q2["janela_fim"] < "2026-10"
q2 = q2[["grupo", "id_produto", "descricao", "mediana3", "janela_fim", "lojas3", "ic3", "lojas6", "ic6", "regra_janela", "referencia_antiga"]].round(2)
salvar(q2, "precisao_por_sku.csv")
print(q2["regra_janela"].value_counts().to_string())

# 5. erro fora da amostra
res = []
for e, g in obs.groupby("id_produto"):
    ms = sorted(g["__mes"].unique()); a3, a1 = [], []
    for i in range(3, len(ms)):
        alvo = g[g["__mes"] == ms[i]].groupby("endereco_completo")["preco_regular"].median()
        r3 = g[g["__mes"].isin(ms[i-3:i])].groupby("endereco_completo")["preco_regular"].median().median()
        r1 = g[g["__mes"] == ms[i-1]].groupby("endereco_completo")["preco_regular"].median().median()
        a3.append(abs(alvo.median() - r3)); a1.append(abs(alvo.median() - r1))
    if a3:
        res.append({"id_produto": e, "meses_testados": len(a3), "err3": np.mean(a3), "err1": np.mean(a1)})
R = pd.DataFrame(res).merge(cat[["id_produto", "grupo", "descricao"]], on="id_produto")
R["melhor"] = np.where(R["err3"] < R["err1"], "3 meses", np.where(R["err3"] > R["err1"], "ultimo mes", "empate"))
med = obs.groupby("id_produto")["preco_regular"].median()
R["err3_pct"] = R["err3"] / R["id_produto"].map(med) * 100
R["err1_pct"] = R["err1"] / R["id_produto"].map(med) * 100
salvar(R.round(3), "mae_por_sku.csv")
print(R["melhor"].value_counts().to_string())
print("erro % | 3m:", round(R["err3_pct"].mean(), 2), "| ultimo mes:", round(R["err1_pct"].mean(), 2))

In [ ]:
fr = q2[q2["regra_janela"]=="baixa precisao"]["id_produto"].tolist()
print("redes classificadas como atacarejo:", sorted(obs.loc[obs["rede"].astype(str).str.contains("atacad|assa", case=False, regex=True), "rede"].unique()))

def ic_rel(v, seed):
    rng = np.random.default_rng(seed)
    ic = np.percentile([np.median(rng.choice(v, len(v), replace=True)) for _ in range(1000)], [2.5, 97.5])
    return (ic[1]-ic[0])/np.median(v)*100

rows = []
for e in fr:
    g = obs[obs["id_produto"]==e]
    ms = sorted(g["__mes"].unique())[-6:]
    w = g[g["__mes"].isin(ms)].copy()
    w["atacarejo"] = w["rede"].astype(str).str.contains("atacad|assa", case=False, regex=True)
    loja = w.groupby(["endereco_completo","atacarejo"])["preco_regular"].median().reset_index()
    v = loja["preco_regular"].values
    med = np.median(v); mad = np.median(np.abs(v - med))
    sem_out = v[np.abs(v - med) <= 3*1.4826*mad] if mad > 0 else v
    var = loja.loc[~loja["atacarejo"], "preco_regular"].values
    rede = w.groupby("rede")["preco_regular"].median().values
    seed = zlib.crc32(e.encode())
    rows.append({"descricao": cat.set_index("id_produto").loc[e, "descricao"][:35], "lojas": len(v), "redes": w["rede"].nunique(),
                 "cv_pct": v.std()/v.mean()*100, "atacarejo_pct": loja["atacarejo"].mean()*100,
                 "ic_6m": ic_rel(v, seed), "atipicos": len(v)-len(sem_out), "ic_sem_atipico": ic_rel(sem_out, seed),
                 "lojas_varejo": len(var), "ic_varejo": ic_rel(var, seed) if len(var) >= 5 else np.nan,
                 "ic_por_rede": ic_rel(rede, seed)})
D = pd.DataFrame(rows).round(1)
print(D.to_string(index=False))

In [ ]:
def ic_rede(g, n, seed):
    ms = sorted(g["__mes"].unique())[-n:]
    v = g[g["__mes"].isin(ms)].groupby("rede")["preco_regular"].median().values
    return ic_rel(v, seed), len(v)
rows = []
for e, g in obs.groupby("id_produto"):
    s = zlib.crc32(e.encode())
    a3, r3 = ic_rede(g, 3, s); a6, r6 = ic_rede(g, 6, s)
    rows.append({"id_produto": e, "redes3": r3, "ic3_rede": a3, "redes6": r6, "ic6_rede": a6})
Rr = pd.DataFrame(rows).merge(q2[["id_produto","descricao","ic3","ic6","regra_janela"]], on="id_produto")
Rr["regra_rede"] = np.where(Rr["ic3_rede"]<=10,"3 meses",np.where(Rr["ic6_rede"]<=10,"6 meses","baixa precisao"))
print(pd.crosstab(Rr["regra_janela"], Rr["regra_rede"]).to_string())
print("baixa precisao por loja:", (Rr["regra_janela"]=="baixa precisao").sum(), "| por rede:", (Rr["regra_rede"]=="baixa precisao").sum())
print(Rr[(Rr["regra_janela"]!="baixa precisao")&(Rr["regra_rede"]=="baixa precisao")][["descricao","ic3","ic3_rede","ic6","ic6_rede"]].round(1).to_string(index=False))

In [ ]:
def n3(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().upper()
    s = re.sub(r"\bC/\s*", "COM ", s); s = re.sub(r"\bPET\b", " ", s); s = re.sub(r"\bA VACUO\b", "VACUO", s)
    return re.sub(r"\s+", " ", s.replace("-", " ")).strip()
b = raw[(raw["preco_regular"]>0) & ~raw["id_produto"].isin(cat["id_produto"])].sort_values("__mes").drop_duplicates(CHAVE_DUP).copy()
b["__mes"] = b["__mes"].astype(str)
b["dn"] = b["descricao"].map(n3)
ult12 = sorted(b["__mes"].unique())[-12:]
w12 = b[b["__mes"].isin(ult12) & ~b["dn"].str.contains("BARATO")]
w12 = w12[w12["id_produto"].map(lambda c: str(c).isdigit() and len(str(c))==13)]
pool = w12.groupby(["id_produto","dn"]).agg(meses=("__mes","nunique"), ultimo=("__mes","max"), redes=("rede","nunique")).reset_index()
pool = pool[(pool["meses"]>=9) & (pool["ultimo"]==ult12[-1])]
rx = {"ARROZ POLIDO 5":"Arroz","ARROZ PARBOILIZADO 5":"Arroz","FEIJAO PRETO 1":"Feijao","FEIJAO CARIOCA 1":"Feijao",
      "CAFE VACUO":"Cafe","REFRIGERANTE":"Refri","AGUA MINERAL":"Agua","MACARRAO COM OVOS":"Macarrao"}
pool = pool[pool["dn"].map(lambda d: any(d.startswith(k) for k in rx))]
rows = []
for _, r in pool.iterrows():
    g = b[(b["id_produto"]==r["id_produto"]) & (b["dn"]==r["dn"])]
    ms = sorted(g["__mes"].unique())[-6:]
    v = g[g["__mes"].isin(ms)].groupby("endereco_completo")["preco_regular"].median().values
    if len(v) >= 5:
        rows.append({"id_produto": r["id_produto"], "descricao": r["dn"][:45], "meses12": r["meses"], "lojas6": len(v), "ic6": ic_rel(v, zlib.crc32(r["id_produto"].encode()))})
P = pd.DataFrame(rows).sort_values("descricao")
print(len(pool), "candidatos no pool |", len(P), "com >=5 lojas em 6 meses")
print(P.round(1).to_string(index=False))

In [ ]:
import unicodedata, re, zlib
import numpy as np
import pandas as pd

def ic_rel(v, seed):
    rng = np.random.default_rng(seed)
    ic = np.percentile([np.median(rng.choice(v, len(v), replace=True)) for _ in range(1000)], [2.5, 97.5])
    return (ic[1]-ic[0])/np.median(v)*100

def n3(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().upper()
    s = re.sub(r"\bC/\s*", "COM ", s); s = re.sub(r"\bPET\b", " ", s); s = re.sub(r"\bA VACUO\b", "VACUO", s)
    return re.sub(r"\s+", " ", s.replace("-", " ")).strip()
b = raw[(raw["preco_regular"]>0) & ~raw["id_produto"].isin(cat["id_produto"])].sort_values("__mes").drop_duplicates(CHAVE_DUP).copy()
b["__mes"] = b["__mes"].astype(str)
b["dn"] = b["descricao"].map(n3)
ult12 = sorted(b["__mes"].unique())[-12:]
w12 = b[b["__mes"].isin(ult12) & ~b["dn"].str.contains("BARATO")]
w12 = w12[w12["id_produto"].map(lambda c: str(c).isdigit() and len(str(c))==13)]
pool = w12.groupby(["id_produto","dn"]).agg(meses=("__mes","nunique"), ultimo=("__mes","max"), redes=("rede","nunique")).reset_index()
pool = pool[(pool["meses"]>=9) & (pool["ultimo"]==ult12[-1])]
rx = {"ARROZ POLIDO 5":"Arroz","ARROZ PARBOILIZADO 5":"Arroz","FEIJAO PRETO 1":"Feijao","FEIJAO CARIOCA 1":"Feijao",
      "CAFE VACUO":"Cafe","REFRIGERANTE":"Refri","AGUA MINERAL":"Agua","MACARRAO COM OVOS":"Macarrao"}
pool = pool[pool["dn"].map(lambda d: any(d.startswith(k) for k in rx))]
rows = []
for _, r in pool.iterrows():
    g = b[(b["id_produto"]==r["id_produto"]) & (b["dn"]==r["dn"])]
    ms = sorted(g["__mes"].unique())[-6:]
    v = g[g["__mes"].isin(ms)].groupby("endereco_completo")["preco_regular"].median().values
    if len(v) >= 5:
        rows.append({"id_produto": r["id_produto"], "descricao": r["dn"][:45], "meses12": r["meses"], "lojas6": len(v), "ic6": ic_rel(v, zlib.crc32(r["id_produto"].encode()))})
P = pd.DataFrame(rows).sort_values("descricao")
print(len(pool), "candidatos no pool |", len(P), "com >=5 lojas em 6 meses")
print(P.round(1).to_string(index=False))

In [ ]:
import pandas as pd

d = pd.read_parquet("raw_data/daily_added_exact_skus.parquet")
print("colunas:", d.columns.tolist())
print(d.dtypes.to_string())
print("linhas:", len(d))
print(d.head(3).to_string())

In [ ]:
import numpy as np
import pandas as pd
from mercado.base_precos import carregar_base, carregar_catalogo, limpar, CHAVE_DUP

raw = carregar_base(); cat = carregar_catalogo(); obs = limpar(raw, cat)
obs["__mes"] = obs["__mes"].astype(str)
d = pd.read_parquet("raw_data/daily_added_exact_skus.parquet")
d["id_produto"] = d["id_produto"].astype(str)

print("linhas:", len(d), "| datas:", d["data_pesquisa"].nunique(), "| source_file:", d["source_file"].nunique())
print("preco_regular > 0 e finito:", (np.isfinite(d["preco_regular"]) & (d["preco_regular"] > 0)).sum())
print("codigos fora do catalogo:", (~d["id_produto"].isin(cat["id_produto"])).sum(), "| SKUs cobertos:", d["id_produto"].nunique())
print("sem id_empresa:", d["id_empresa"].isna().sum() + (d["id_empresa"].astype(str).str.strip() == "").sum())

print("duplicatas dentro do adicional (chave):", d.duplicated(CHAVE_DUP).sum())
chave_obs = obs[CHAVE_DUP].drop_duplicates().assign(ja_temos=True)
m = d.merge(chave_obs, on=CHAVE_DUP, how="left")
print("ja existem na nossa base (mesma chave):", m["ja_temos"].fillna(False).astype(bool).sum())

# lojas: novas vs ja vistas na janela de 6 meses da nossa base
ult6 = sorted(obs["__mes"].unique())[-6:]
lojas_old = set(obs.loc[obs["__mes"].isin(ult6), "endereco_completo"])
lojas_new = set(d["endereco_completo"])
print("lojas na nossa janela de 6 meses:", len(lojas_old), "| lojas no adicional:", len(lojas_new),
      "| em comum:", len(lojas_old & lojas_new), "| so no adicional:", len(lojas_new - lojas_old))
print("lojas distintas somando as duas fontes:", len(lojas_old | lojas_new))
print("preco_regular por data (n linhas):"); print(d.groupby("data_pesquisa").size().to_string())

In [ ]:
import zlib
def ic_rel(v, seed):
    rng = np.random.default_rng(seed)
    ic = np.percentile([np.median(rng.choice(v, len(v), replace=True)) for _ in range(1000)], [2.5, 97.5])
    return (ic[1]-ic[0])/np.median(v)*100

d2 = d.copy(); d2["__mes"] = d2["date"].dt.to_period("M").astype(str)
cols = ["id_produto","endereco_completo","__mes","preco_regular"]
novo = pd.concat([obs[cols], d2[cols]], ignore_index=True)

def ic_janela(g, n, seed):
    ms = sorted(g["__mes"].unique())[-n:]
    v = g[g["__mes"].isin(ms)].groupby("endereco_completo")["preco_regular"].median().values
    return ic_rel(v, seed), len(v)

rows = []
for e in cat["id_produto"]:
    s = zlib.crc32(e.encode())
    a = obs[obs["id_produto"]==e]; b = novo[novo["id_produto"]==e]
    ia, la = ic_janela(a, 6, s); ib, lb = ic_janela(b, 6, s)
    rows.append({"id_produto": e, "lojas_antes": la, "ic6_antes": ia, "lojas_depois": lb, "ic6_depois": ib})
T = pd.DataFrame(rows).merge(cat[["id_produto","descricao"]], on="id_produto")
fr = q2[q2["regra_janela"]=="baixa precisao"]["id_produto"]
print("IC6>10 antes:", (T["ic6_antes"]>10).sum(), "| depois:", (T["ic6_depois"]>10).sum())
print(T[T["id_produto"].isin(fr)][["descricao","lojas_antes","ic6_antes","lojas_depois","ic6_depois"]].round(1).to_string(index=False))

In [ ]:
print(pd.crosstab(T["ic6_antes"] > 10, T["ic6_depois"] > 10, rownames=["fraco antes"], colnames=["fraco depois"]).to_string())
print(T[(T["ic6_antes"] <= 10) & (T["ic6_depois"] > 10)][["descricao", "lojas_antes", "ic6_antes", "lojas_depois", "ic6_depois"]].round(1).to_string(index=False))
print("lojas medias por SKU | antes:", round(T["lojas_antes"].mean(), 1), "| depois:", round(T["lojas_depois"].mean(), 1))

In [ ]:
rows = []
for e in cat["id_produto"]:
    s = zlib.crc32(e.encode()); b = novo[novo["id_produto"]==e]
    i3, l3 = ic_janela(b, 3, s); i6, l6 = ic_janela(b, 6, s)
    rows.append({"id_produto": e, "lojas3": l3, "ic3": i3, "lojas6": l6, "ic6": i6})
Z = pd.DataFrame(rows).merge(cat[["id_produto","grupo","descricao"]], on="id_produto")
Z["regra_janela"] = np.where(Z["ic3"]<=10,"3 meses",np.where(Z["ic6"]<=10,"6 meses","baixa precisao"))
Z.round(2).to_csv("raw_data/precisao_com_outubro_diario.csv", index=False)
print(Z["regra_janela"].value_counts().to_string())
print(Z[Z["regra_janela"]=="baixa precisao"][["descricao","lojas3","ic3","lojas6","ic6"]].round(1).to_string(index=False))

In [ ]:
import pandas as pd
Z = pd.read_csv("raw_data/precisao_com_outubro_diario.csv", dtype={"id_produto": str})
Z["janela_usada"] = Z["regra_janela"].map({"3 meses": 3, "6 meses": 6})
Z["margem_erro_pct"] = Z.apply(lambda r: r["ic3"] if r["regra_janela"] == "3 meses" else r["ic6"], axis=1) / 2
Z["lojas_distintas"] = Z.apply(lambda r: r["lojas3"] if r["regra_janela"] == "3 meses" else r["lojas6"], axis=1)
Z["confiavel"] = Z["regra_janela"].map(lambda x: "NAO (baixa precisao)" if x == "baixa precisao" else "SIM")
Z["ordem"] = (Z["regra_janela"] == "baixa precisao").astype(int)
T = Z.sort_values(["ordem", "margem_erro_pct"])[["grupo", "descricao", "margem_erro_pct", "lojas_distintas", "janela_usada", "confiavel"]].copy()
T["margem_erro_pct"] = T["margem_erro_pct"].round(1)
T["janela_usada"] = T["janela_usada"].fillna(6).astype(int)
print(T.to_csv(sep="\t", index=False))

In [ ]:
import pandas as pd
from mercado.base_precos import carregar_base, carregar_catalogo, limpar

raw = carregar_base(); cat = carregar_catalogo(); obs = limpar(raw, cat)
obs["__mes"] = obs["__mes"].astype(str)
d = pd.read_parquet("raw_data/daily_added_exact_skus.parquet")
d["id_produto"] = d["id_produto"].astype(str)
d["__mes"] = d["date"].dt.to_period("M").astype(str)
cols = ["id_produto", "endereco_completo", "__mes", "preco_regular"]
todas = pd.concat([obs[cols], d[cols]], ignore_index=True)

Z = pd.read_csv("raw_data/precisao_com_outubro_diario.csv", dtype={"id_produto": str})
Z["janela_usada"] = Z["regra_janela"].map({"3 meses": 3, "6 meses": 6}).fillna(6).astype(int)
Z["margem_erro_pct"] = (Z.apply(lambda r: r["ic3"] if r["regra_janela"] == "3 meses" else r["ic6"], axis=1) / 2).round(1)
Z["lojas_distintas"] = Z.apply(lambda r: r["lojas3"] if r["regra_janela"] == "3 meses" else r["lojas6"], axis=1)

def n_janela(r):
    g = todas[todas["id_produto"] == r["id_produto"]]
    ms = sorted(g["__mes"].unique())[-int(r["janela_usada"]):]
    return int(g["__mes"].isin(ms).sum())
Z["ocorrencias_janela"] = Z.apply(n_janela, axis=1)
Z["ocorrencias_total"] = Z["id_produto"].map(todas.groupby("id_produto").size()).astype(int)
Z["confiavel"] = Z["regra_janela"].map(lambda x: "NAO (baixa precisao)" if x == "baixa precisao" else "SIM")
Z["observacao"] = ""
Z.loc[Z["grupo"] == "Cerveja", "observacao"] = "Referencia antiga: ultimo dado em fev/2026"
Z["ordem"] = (Z["regra_janela"] == "baixa precisao").astype(int)
T = Z.sort_values(["ordem", "margem_erro_pct"])[["grupo", "descricao", "margem_erro_pct", "lojas_distintas", "ocorrencias_janela", "ocorrencias_total", "janela_usada", "confiavel", "observacao"]]
print(T.shape, "| total ocorrencias:", T["ocorrencias_total"].sum())
print(T.to_csv(sep="\t", index=False))